# FLAIR (clean version)

Cleaned-up copy of the FLAIR Colab notebook (original: https://github.com/conservation-technology-group/FLAIR).
Behaviour is unchanged; this version only removes dead code, moves every path and setting into **one config cell**,
and fails early with a clear message when something is wrong, so that **Runtime > Run all** works start to end.

**Before you run:** use a GPU runtime, and check the paths/settings in the *Configuration* cell.

**Where things go**
- Fast local scratch (lost on disconnect): `WORK_DIR` - frames, intermediate masks, overlay frames.
- Persistent results: `PROJECT_DIR/runs/<RUN_NAME>/` on Drive - masks (zipped), boxes, tracked video, `run_info.json` (config, timings, environment).
- Nothing outside `PROJECT_DIR/runs/<RUN_NAME>/` is ever written to or deleted on Drive. The original notebook's outputs are only *read*, for the optional regression check at the end.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running on Colab - skipping Drive mount")

# Configuration

In [ ]:
import json
import os
import shutil
from pathlib import Path

# =============================== EDIT THIS CELL ===============================
# --- Locations -----------------------------------------------------------------
PROJECT_DIR  = "/content/drive/MyDrive/DS340WCourseProject"                    # persistent (Drive)
SOURCE_VIDEO = f"{PROJECT_DIR}/FLAIR_code/FLAIR/example_video_blacktip.mp4"    # input video (on Drive)
WORK_DIR     = "/content/flair_work"                                           # fast local scratch

RUN_NAME        = "dev"    # results go to PROJECT_DIR/runs/<RUN_NAME>/. Use a new name for runs you want to keep.
OVERWRITE       = True     # True: replace an existing Drive run folder of this name. False: stop if it exists.
FORCE_REEXTRACT = False    # True: re-extract video frames even if they are already cached locally

# Optional: pruned masks from the original notebook, used (read-only) for the regression check at the end.
BASELINE_PRUNED_DIR = f"{PROJECT_DIR}/video_segments_example_video_pruned"
REGRESSION_MIN_DICE = 0.95

# --- Video clip to process ---------------------------------------------------------
CLIP_START = "00:00:00"
CLIP_END   = "00:00:08"    # None = whole video

# --- SAM2 ----------------------------------------------------------------------------
MODEL_CFG     = "sam2_hiera_l.yaml"
VOS_OPTIMIZED = False      # can be True on an appropriate GPU
_LEGACY_CKPT  = "/content/drive/MyDrive/checkpoints/sam2_hiera_large.pt"   # where the original notebook put it
CHECKPOINT    = _LEGACY_CKPT if os.path.exists(_LEGACY_CKPT) else f"{PROJECT_DIR}/checkpoints/sam2_hiera_large.pt"

MASK_GENERATOR_PARAMS = dict(
    points_per_side=32,
    points_per_batch=32,
    pred_iou_thresh=0.7,
    stability_score_thresh=0.92,
    stability_score_offset=0.7,
    crop_n_layers=1,
    box_nms_thresh=0.3,
    crop_nms_thresh=0.3,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=0,
    use_m2m=False,
)

# --- FLAIR parameters ----------------------------------------------------------------
RUN_EVERY_N_FRAMES = 30    # run automatic mask generation every N frames (typically the video fps)
MIN_MASK_LENGTH    = 75    # minimum mask bbox width/height in pixels (calibrate from the grid plot below)
MAX_MASK_LENGTH    = 300   # maximum mask bbox width/height in pixels (calibrate from the grid plot below)
BBOX_BUFFER        = 25    # padding around bounding boxes
WINDOW_LENGTH      = 3     # window length for SAM2 video propagation

# --- CLIP ----------------------------------------------------------------------------
CLIP_CONFIDENCE = 0.95     # confidence threshold for CLIP filtering
CLIP_PROMPTS = [
    "A drone image of white shape on surface of the ocean",
    "A clear zoomed-in drone image of a complete dark nurse shark in the ocean with distinct pectoral fins and tail",
    "A drone image of the ocean with no distinct sharks",
    "A drone image of a rock under the surface of the ocean with no sharks",
    "A drone image of sand in the ocean",
    "A drone image of rocks on land",
]
CLIP_CORRECT = [1]         # indices of the prompts above that describe the object you want to keep

# --- Mask review / pruning -------------------------------------------------------------
N_PER_MASK    = 9          # images per track in the review grid (3x3)
RESIZE_FACTOR = 0.3        # shrink review images to fit
KEEP_KEYS     = [2]        # track IDs that are true positives (read them off the review grid, then edit here)
# =============================== END OF SETTINGS ===============================

# ---- Derived paths and early checks (do not edit) -------------------------------------
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in (".", ".."):
    raise ValueError(f"RUN_NAME must be a simple folder name, got {RUN_NAME!r}")
if not os.path.isfile(SOURCE_VIDEO):
    raise FileNotFoundError(
        f"Source video not found: {SOURCE_VIDEO}\nCheck PROJECT_DIR / SOURCE_VIDEO in this cell "
        f"and that Google Drive mounted correctly."
    )

FRAMES_DIR   = f"{WORK_DIR}/frames/{Path(SOURCE_VIDEO).stem}"
RUN_LOCAL    = f"{WORK_DIR}/runs/{RUN_NAME}"
RUN_DRIVE    = f"{PROJECT_DIR}/runs/{RUN_NAME}"
SEGMENTS_DIR = f"{RUN_LOCAL}/video_segments"
PRUNED_DIR   = f"{RUN_LOCAL}/video_segments_pruned"
OVERLAY_DIR  = f"{RUN_LOCAL}/tracked_frames"
OUTPUT_TXT   = f"{RUN_LOCAL}/keep_boxes.txt"
OUTPUT_VIDEO = f"{RUN_LOCAL}/tracked_video.mp4"

if os.path.exists(RUN_DRIVE) and not OVERWRITE:
    raise FileExistsError(
        f"Run folder already exists: {RUN_DRIVE}\n"
        f"Set OVERWRITE = True to replace it, or choose a new RUN_NAME."
    )

# Local run folder always starts clean (frames are cached separately in FRAMES_DIR).
shutil.rmtree(RUN_LOCAL, ignore_errors=True)
os.makedirs(RUN_LOCAL)

_CONFIG_KEYS = [
    "PROJECT_DIR", "SOURCE_VIDEO", "WORK_DIR", "RUN_NAME", "OVERWRITE", "CLIP_START", "CLIP_END",
    "MODEL_CFG", "VOS_OPTIMIZED", "CHECKPOINT", "MASK_GENERATOR_PARAMS", "RUN_EVERY_N_FRAMES",
    "MIN_MASK_LENGTH", "MAX_MASK_LENGTH", "BBOX_BUFFER", "WINDOW_LENGTH", "CLIP_CONFIDENCE",
    "CLIP_PROMPTS", "CLIP_CORRECT", "N_PER_MASK", "RESIZE_FACTOR", "KEEP_KEYS",
]
CONFIG = {k: globals()[k] for k in _CONFIG_KEYS}

print(f"Source video : {SOURCE_VIDEO}")
print(f"Checkpoint   : {CHECKPOINT}")
print(f"Frames (local, cached): {FRAMES_DIR}")
print(f"Run (local)  : {RUN_LOCAL}")
print(f"Run (Drive)  : {RUN_DRIVE}  (written at the sync steps)")

# Load dependencies

In [ ]:
import importlib.metadata as importlib_metadata
import importlib.util
import subprocess


def _run_shell(cmd):
    """Run a shell command quietly; on failure raise with the tail of its output."""
    result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        tail = "\n".join(result.stdout.splitlines()[-25:])
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}\n{tail}")


def _torch_pinned():
    try:
        return importlib_metadata.version("torch").startswith("2.6.0")
    except importlib_metadata.PackageNotFoundError:
        return False


if _torch_pinned() and importlib.util.find_spec("sam2") and importlib.util.find_spec("transformers"):
    print("Dependencies already installed - skipping.")
else:
    print("Installing dependencies (a few minutes)...")
    _run_shell("apt-get update -qq && apt-get install -y -qq gcc g++")
    _run_shell("pip install -q numpy matplotlib pillow tqdm opencv-python")
    _run_shell("pip install -q torch==2.6.0 torchvision==0.21.0")
    _run_shell("pip install -q transformers==4.51.3 accelerate==1.7.0 omegaconf==2.3.0 iopath==0.1.10 "
               "huggingface-hub==0.31.4 regex==2024.11.6 portalocker==3.1.1")
    # SAM2 is built from source. If you see "sam2/_C.so: undefined symbol" later, SAM2's optional hole-filling
    # post-processing is being skipped (results are otherwise fine). Adding --no-build-isolation may fix it.
    _run_shell("pip install -q git+https://github.com/conservation-technology-group/sam2.git")
    print("Done. If Colab asks you to restart the runtime, restart it and Run all again.")

In [ ]:
import gc
import importlib.metadata as importlib_metadata
import random
import subprocess
import time
import urllib.request
import zipfile
from collections import defaultdict
from contextlib import contextmanager

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from PIL import Image
from tqdm import tqdm

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab: Runtime > Change runtime type > pick a GPU, then Run all again.")
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()   # SAM2 runs under bfloat16 autocast; enabled once


# ---------------------------------------------------------------- generic helpers
timings = {}


@contextmanager
def timed(name):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        timings[name] = round(time.perf_counter() - t0, 2)
        print(f"[time] {name}: {timings[name]} s")


def run_cmd(args):
    """Run a command (list form); on failure raise with the tail of its output."""
    result = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        tail = "\n".join(result.stdout.splitlines()[-25:])
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, args))}\n{tail}")
    return result.stdout


def list_frames(frames_dir):
    """Frame image names, sorted numerically (frames must be named 00000.jpg, 00001.jpg, ...)."""
    return sorted(
        [p for p in os.listdir(frames_dir) if os.path.splitext(p)[-1].lower() in (".jpg", ".jpeg")],
        key=lambda p: int(os.path.splitext(p)[0]),
    )


def probe_fps(video_path):
    """Frames per second of a video via ffprobe (None if unavailable)."""
    try:
        out = run_cmd(["ffprobe", "-v", "error", "-select_streams", "v:0",
                       "-show_entries", "stream=r_frame_rate", "-of", "csv=p=0", video_path]).strip()
        num, _, den = out.partition("/")
        return float(num) / float(den or 1)
    except Exception:
        return None


# ---------------------------------------------------------------- mask / box helpers
def keep_masks(all_frames_masks, min_len, max_len):
    """Keep masks whose bbox width AND height are both between min_len and max_len."""
    keep = [
        i for i, m in enumerate(all_frames_masks)
        if min_len < m['bbox'][2] < max_len and min_len < m['bbox'][3] < max_len
    ]
    return [all_frames_masks[j] for j in keep]


def xywh_to_xyxy(bbox):
    return [int(bbox[0]), int(bbox[1]), int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3])]


def bboxes_from_masks(masks, image_shape, buffer):
    """xyxy boxes around masks, padded by `buffer` pixels and clipped to the image."""
    bounding_boxes = []
    for m in masks:
        x_min, y_min, x_max, y_max = xywh_to_xyxy(m['bbox'])
        bounding_boxes.append([
            max(0, x_min - buffer),
            max(0, y_min - buffer),
            min(x_max + buffer, image_shape[1]),
            min(y_max + buffer, image_shape[0]),
        ])
    return bounding_boxes


def get_bounding_box(mask):
    """(x_min, y_min, x_max, y_max) of a boolean mask, or None if empty."""
    mask = np.squeeze(mask)
    y_idx, x_idx = np.where(mask)
    if not x_idx.size or not y_idx.size:
        return None
    return np.min(x_idx), np.min(y_idx), np.max(x_idx), np.max(y_idx)


def calculate_iou(b1, b2):
    xA, yA, xB, yB = max(b1[0], b2[0]), max(b1[1], b2[1]), min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    union = (b1[2] - b1[0]) * (b1[3] - b1[1]) + (b2[2] - b2[0]) * (b2[3] - b2[1]) - inter
    return inter / union if union > 0 else 0


def mask_to_polygons(mask):
    mask = np.squeeze(mask.astype(np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for contour in contours:
        if len(contour) >= 3:
            polygons.append(contour.reshape(-1, 2).tolist())
    return polygons


def polygons_to_mask(polygons, image_shape):
    mask = np.zeros(image_shape[:2], dtype=np.uint8)
    for polygon in polygons:
        if len(polygon) >= 3:
            cv2.fillPoly(mask, [np.array(polygon, dtype=np.int32)], 1)
    return mask.astype(bool)


# ---------------------------------------------------------------- saving results to Drive
_drive_initialised = False


def _zip_dir(src_dir, dst_zip):
    """Zip a flat folder of .npy files into one archive (one Drive write instead of hundreds) and verify it."""
    names = sorted(f for f in os.listdir(src_dir) if f.endswith(".npy"))
    with zipfile.ZipFile(dst_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for f in names:
            z.write(os.path.join(src_dir, f), arcname=f)
    with zipfile.ZipFile(dst_zip) as z:
        if z.testzip() is not None or len(z.namelist()) != len(names):
            raise RuntimeError(f"Archive verification failed: {dst_zip}")
    return len(names)


def sync_run_to_drive(stage):
    """Copy this run's results to PROJECT_DIR/runs/<RUN_NAME>/ on Drive. Safe to call repeatedly.

    Only ever touches RUN_DRIVE. With OVERWRITE=True the old folder is replaced the first time this is
    called in a session (i.e. only once new results exist)."""
    global _drive_initialised
    if os.path.dirname(os.path.realpath(RUN_DRIVE)) != os.path.realpath(f"{PROJECT_DIR}/runs"):
        raise RuntimeError(f"Refusing to write outside {PROJECT_DIR}/runs: {RUN_DRIVE}")
    if not _drive_initialised:
        if os.path.exists(RUN_DRIVE):
            if not OVERWRITE:
                raise FileExistsError(f"Run folder already exists: {RUN_DRIVE} (set OVERWRITE = True or change RUN_NAME)")
            shutil.rmtree(RUN_DRIVE)
        _drive_initialised = True
    os.makedirs(RUN_DRIVE, exist_ok=True)

    saved = {}
    for src_dir, zip_name in [(SEGMENTS_DIR, "video_segments.zip"), (PRUNED_DIR, "video_segments_pruned.zip")]:
        if os.path.isdir(src_dir) and any(f.endswith(".npy") for f in os.listdir(src_dir)):
            saved[zip_name] = _zip_dir(src_dir, f"{RUN_DRIVE}/{zip_name}")
    for path in (OUTPUT_TXT, OUTPUT_VIDEO):
        if os.path.isfile(path):
            shutil.copy2(path, RUN_DRIVE)
            saved[os.path.basename(path)] = os.path.getsize(path)

    info = {
        "stage": stage,
        "config": CONFIG,
        "timings_s": timings,
        "regression": globals().get("regression_result"),
        "environment": {
            "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0),
            "transformers": importlib_metadata.version("transformers"),
        },
    }
    with open(f"{RUN_DRIVE}/run_info.json", "w") as f:
        json.dump(info, f, indent=2, default=str)
    print(f"[sync:{stage}] saved to {RUN_DRIVE}: {saved}")

# SAM2 setup

In [ ]:
def download_checkpoint_if_missing(path):
    """Download the SAM2 Large checkpoint; written via a temp file so an interrupted download is never mistaken for a good one."""
    if os.path.exists(path):
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print(f"Downloading SAM2 checkpoint to {path} ...")
    tmp_path = path + ".part"
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt", tmp_path)
    os.replace(tmp_path, path)
    print("Download complete.")


from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

download_checkpoint_if_missing(CHECKPOINT)
device = torch.device("cuda")

# Video predictor (tracking/propagation) and automatic mask generator (candidate masks)
predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT, device=device, vos_optimized=VOS_OPTIMIZED)
_sam = build_sam2(MODEL_CFG, CHECKPOINT, device=device, apply_postprocessing=False)
mask_generator = SAM2AutomaticMaskGenerator(model=_sam, **MASK_GENERATOR_PARAMS)
print("SAM2 loaded on", device)

# Extract video frames

Frames are written to local scratch and reused if the source video and clip settings have not changed.

In [ ]:
def extract_frames():
    meta = {"source": SOURCE_VIDEO, "mtime": os.path.getmtime(SOURCE_VIDEO), "start": CLIP_START, "end": CLIP_END}
    marker = Path(FRAMES_DIR) / ".extract_meta.json"
    if not FORCE_REEXTRACT and marker.exists() and list_frames(FRAMES_DIR):
        try:
            if json.loads(marker.read_text()) == meta:
                print(f"Using cached frames in {FRAMES_DIR} ({len(list_frames(FRAMES_DIR))} frames)")
                return
        except ValueError:
            pass
    shutil.rmtree(FRAMES_DIR, ignore_errors=True)
    os.makedirs(FRAMES_DIR)
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"]
    if CLIP_START:
        cmd += ["-ss", CLIP_START]
    if CLIP_END:
        cmd += ["-to", CLIP_END]
    cmd += ["-i", SOURCE_VIDEO, "-q:v", "2", "-start_number", "0", f"{FRAMES_DIR}/%05d.jpg"]
    run_cmd(cmd)
    n_frames = len(list_frames(FRAMES_DIR))
    if n_frames == 0:
        raise RuntimeError(f"ffmpeg produced no frames from {SOURCE_VIDEO} (check CLIP_START / CLIP_END)")
    marker.write_text(json.dumps(meta))
    print(f"Extracted {n_frames} frames to {FRAMES_DIR}")


with timed("extract_frames"):
    extract_frames()

frame_names = list_frames(FRAMES_DIR)
SOURCE_FPS = probe_fps(SOURCE_VIDEO)
if SOURCE_FPS and abs(SOURCE_FPS - RUN_EVERY_N_FRAMES) > 1:
    print(f"Note: source video is {SOURCE_FPS:.2f} fps but RUN_EVERY_N_FRAMES={RUN_EVERY_N_FRAMES}; "
          f"'Second N' labels below are every {RUN_EVERY_N_FRAMES} frames, not true seconds.")

# One frame every RUN_EVERY_N_FRAMES frames is used for candidate-mask generation
images = [
    np.array(Image.open(os.path.join(FRAMES_DIR, frame_names[i])).convert("RGB"))
    for i in range(0, len(frame_names), RUN_EVERY_N_FRAMES)
]
image_shape = images[0].shape
print(f"{len(frame_names)} frames, {len(images)} sampled for mask generation, frame size {image_shape[1]}x{image_shape[0]}")

Estimate the minimum and maximum mask size from the gridded frame below (the object should fit within an n x n pixel box).
If uncertain, set the minimum to 0 and the maximum to your estimated max. Edit `MIN_MASK_LENGTH` / `MAX_MASK_LENGTH` in the
configuration cell and Run all again if they need changing.

In [ ]:
# Show a sampled frame with a transparent grid every 100 pixels
second = min(1, len(images) - 1)
plt.figure(figsize=(9, 6))
plt.title(f"Second {second}")
plt.imshow(images[second])

height, width = images[0].shape[:2]
for x in range(0, width, 100):
    plt.axvline(x=x, color='white', linewidth=0.8, alpha=0.5)
for y in range(0, height, 100):
    plt.axhline(y=y, color='white', linewidth=0.8, alpha=0.5)

plt.show()
gc.collect()

# SAM2 automatic mask generation

Generate candidate masks on each sampled frame, keep those in the allowed size range, and turn them into padded boxes.

In [ ]:
all_frames_bboxes = []
with timed("mask_generation"):
    for img in tqdm(images):
        masks = mask_generator.generate(img)
        masks = keep_masks(masks, MIN_MASK_LENGTH, MAX_MASK_LENGTH)
        all_frames_bboxes.append(bboxes_from_masks(masks, img.shape, BBOX_BUFFER))
        torch.cuda.empty_cache()
print("Candidate boxes per sampled frame:", [len(b) for b in all_frames_bboxes])

# CLIP filtering

Keep only candidate boxes that CLIP classifies as the target. Prompts are set in the configuration cell; some manual prompt tuning may be necessary.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32", use_fast=False)

In [ ]:
keep_boxes = []   # (sampled-frame index, box index within that frame, [x1, y1, x2, y2])

with timed("clip_filtering"):
    for i, (img, bboxes) in enumerate(zip(images, all_frames_bboxes)):
        crops = [img[y1:y2 + 1, x1:x2 + 1] for x1, y1, x2, y2 in bboxes]
        if not crops:
            continue

        inputs = clip_processor(text=CLIP_PROMPTS, images=crops, return_tensors="pt", padding=True)
        outputs = clip_model(**inputs)
        probs = outputs["logits_per_image"].softmax(dim=1).detach().numpy()

        for j, p in enumerate(probs):
            if (p[CLIP_CORRECT] > CLIP_CONFIDENCE).any():
                keep_boxes.append((i, j, bboxes[j]))

                # Show the accepted crop
                print("Second", i)
                plt.imshow(crops[j])
                plt.axis('off')
                plt.show()
                plt.close()
                gc.collect()

if not keep_boxes:
    raise RuntimeError(
        "CLIP kept 0 boxes. Try lowering CLIP_CONFIDENCE, adjusting CLIP_PROMPTS / CLIP_CORRECT, "
        "or widening MIN_MASK_LENGTH / MAX_MASK_LENGTH."
    )

with open(OUTPUT_TXT, "w") as f:
    for i, j, bbox in keep_boxes:
        f.write(f"{i},{j},{','.join(map(str, bbox))}\n")
print(f"Saved {len(keep_boxes)} boxes to {OUTPUT_TXT}")

# Group by sampled frame
frame_keep_boxes = [[] for _ in range(len(images))]
for i, j, bbox in keep_boxes:
    frame_keep_boxes[i].append((i, j, bbox))

# SAM2 video propagation + frame alignment

In [ ]:
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    inference_state = predictor.init_state(video_path=FRAMES_DIR, async_loading_frames=True)
print("Inference state ready")

In [ ]:
def _propagate(predictor, inference_state, output_dir, start, max_frame, image_shape):
    """Propagate prompted objects through the video and merge the resulting polygons into frame_<idx>.npy files."""
    all_results = defaultdict(dict)
    with torch.autocast("cuda", torch.bfloat16):
        with torch.inference_mode():
            iterator = predictor.propagate_in_video(
                inference_state,
                start_frame_idx=start,
                max_frame_num_to_track=max_frame,
            )
            for out_frame_idx, out_obj_ids, out_mask_logits in iterator:
                frame_segments = {
                    out_obj_id: mask_to_polygons((out_mask_logits[i] > 0.0).cpu().numpy())
                    for i, out_obj_id in enumerate(out_obj_ids)
                }
                all_results[out_frame_idx].update(frame_segments)
                del frame_segments, out_obj_ids, out_mask_logits

    for out_frame_idx, new_segments in all_results.items():
        file_path = os.path.join(output_dir, f'frame_{out_frame_idx}.npy')
        try:
            old_segments = np.load(file_path, allow_pickle=True).item()
            old_segments.update(new_segments)
            np.save(file_path, old_segments)
            del old_segments
        except FileNotFoundError:
            np.save(file_path, new_segments)
        del new_segments


def run_propagation(predictor, inference_state, frame_keep_boxes, image_shape, frame_gap,
                    window_length, output_dir, bbox_buffer):
    """FLAIR tracking: prompt SAM2 with CLIP-approved boxes, propagate over a window, de-duplicate and re-propagate."""
    os.makedirs(output_dir, exist_ok=True)
    start_time = time.time()
    curr_id = 0
    correct = []
    frames = []
    repropagate = []

    for a in range(len(frame_keep_boxes)):
        gc.collect()
        torch.cuda.empty_cache()

        print(f"\nSecond: {a}")

        if curr_id == 0:
            predictor.reset_state(inference_state)
            for b in range(len(frame_keep_boxes[a])):
                curr_id += 1
                i, j, bbox = frame_keep_boxes[a][b]
                predictor.add_new_points_or_box(
                    inference_state=inference_state,
                    frame_idx=i * frame_gap,
                    obj_id=curr_id,
                    box=bbox,
                )
                correct.append(False)
                frames.append(i)
                repropagate.append(i + window_length)
            if curr_id != 0:
                _propagate(
                    predictor,
                    inference_state,
                    output_dir,
                    start=max(0, frame_gap * (a - window_length)),
                    max_frame=(frame_gap * window_length) + (frame_gap * a - max(0, frame_gap * (a - window_length))),
                    image_shape=image_shape,
                )

        else:
            prev_curr_id = curr_id
            predictor.reset_state(inference_state)

            for b in range(len(frame_keep_boxes[a])):
                i, j, bbox = frame_keep_boxes[a][b]
                filename = os.path.join(output_dir, f'frame_{i * frame_gap}.npy')
                mask_add = True

                if os.path.exists(filename):
                    frame_segments_poly = np.load(filename, allow_pickle=True).item()
                    frame_segments = {k: polygons_to_mask(v, image_shape) for k, v in frame_segments_poly.items()}
                    for key in sorted(frame_segments.keys(), key=int):
                        bounding_box = get_bounding_box(frame_segments[key])
                        if bounding_box is not None:
                            bounding_box = [
                                max(0, bounding_box[0] - bbox_buffer),
                                max(0, bounding_box[1] - bbox_buffer),
                                min(bounding_box[2] + bbox_buffer, image_shape[1]),
                                min(bounding_box[3] + bbox_buffer, image_shape[0]),
                            ]
                            iou = calculate_iou(bbox, bounding_box)
                            if iou > 0.7:
                                print("Duplicate mask!")
                                mask_add = False
                                if i != frames[int(key) - 1]:
                                    correct[int(key) - 1] = True
                                break
                    del frame_segments

                if mask_add:
                    curr_id += 1
                    predictor.add_new_points_or_box(
                        inference_state=inference_state,
                        frame_idx=i * frame_gap,
                        obj_id=curr_id,
                        box=bbox,
                    )
                    correct.append(False)
                    frames.append(i)
                    repropagate.append(i + window_length)

            if prev_curr_id != curr_id:
                _propagate(
                    predictor,
                    inference_state,
                    output_dir,
                    start=max(0, frame_gap * (a - window_length)),
                    max_frame=(frame_gap * window_length) + (frame_gap * a - max(0, frame_gap * (a - window_length))),
                    image_shape=image_shape,
                )

        # Re-propagate tracks that were confirmed by a later detection
        indices = [i for i, x in enumerate(repropagate) if x == a]
        should_reprop = False
        predictor.reset_state(inference_state)

        for index in indices:
            if correct[index]:
                filename = os.path.join(output_dir, f'frame_{(frame_gap * a) - 1}.npy')
                frame_segments_old_poly = np.load(filename, allow_pickle=True).item()
                frame_segments_old = {k: polygons_to_mask(v, image_shape) for k, v in frame_segments_old_poly.items()}
                bounding_box = get_bounding_box(frame_segments_old.get(index + 1))
                del frame_segments_old

                if bounding_box is not None:
                    should_reprop = True
                    print("Re-propagating index", index, "at second", a)
                    bounding_box = [
                        max(0, bounding_box[0] - bbox_buffer),
                        max(0, bounding_box[1] - bbox_buffer),
                        min(bounding_box[2] + bbox_buffer, image_shape[1]),
                        min(bounding_box[3] + bbox_buffer, image_shape[0]),
                    ]

                    predictor.add_new_points_or_box(
                        inference_state=inference_state,
                        frame_idx=(frame_gap * a),
                        obj_id=index + 1,
                        box=bounding_box,
                    )
                    repropagate[index] += window_length

        if should_reprop:
            _propagate(
                predictor,
                inference_state,
                output_dir,
                start=frame_gap * a,
                max_frame=frame_gap * window_length,
                image_shape=image_shape,
            )

    # Keep only tracks that were confirmed by a second detection
    correct_indices = [i + 1 for i, is_correct in enumerate(correct) if is_correct]
    npy_files = [f for f in os.listdir(output_dir) if f.endswith(".npy")]
    for fname in tqdm(npy_files, desc="Filtering files"):
        fpath = os.path.join(output_dir, fname)
        data = np.load(fpath, allow_pickle=True).item()
        filtered = {k: v for k, v in data.items() if int(k) in correct_indices}
        np.save(fpath, filtered)

    print(f"Window {window_length} took {time.time() - start_time:.2f} seconds")

In [ ]:
# Propagate masks through the video (a few minutes on a GPU)
with timed("propagation"):
    run_propagation(
        predictor=predictor,
        inference_state=inference_state,
        frame_keep_boxes=frame_keep_boxes,
        image_shape=image_shape,
        frame_gap=RUN_EVERY_N_FRAMES,
        window_length=WINDOW_LENGTH,
        output_dir=SEGMENTS_DIR,
        bbox_buffer=BBOX_BUFFER,
    )

if not any(f.endswith(".npy") for f in os.listdir(SEGMENTS_DIR)):
    raise RuntimeError("Propagation produced no masks. Check the CLIP/size settings above.")

sync_run_to_drive("after_propagation")   # save the expensive step to Drive straight away

# Review tracks

Each track (mask ID) is shown with example frames. Note which IDs are true positives and set `KEEP_KEYS` in the configuration cell.

In [ ]:
def display_mask_grid(n_per_mask, resize_factor, mask_dir, frames_dir, frame_names, image_shape):
    """Show n_per_mask example frames (with the mask's bounding box) for every track."""
    all_samples = defaultdict(list)
    for fname in tqdm(sorted(os.listdir(mask_dir)), desc="Indexing masks"):
        if not fname.endswith(".npy"):
            continue
        frame_index = int(fname.split("_")[1].split(".")[0])
        data = np.load(os.path.join(mask_dir, fname), allow_pickle=True).item()
        for mask_num, mask_polygons in data.items():
            mask = polygons_to_mask(mask_polygons, image_shape)
            bbox = get_bounding_box(mask)
            if bbox is not None:
                all_samples[mask_num].append((frame_index, mask, bbox))
        del data
        gc.collect()

    if not all_samples:
        print("No tracks to display.")
    for mask_num, samples in all_samples.items():
        if len(samples) < n_per_mask:
            continue
        selected = random.sample(samples, n_per_mask)
        frame_indices = [f for f, _, _ in samples]

        fig, axes = plt.subplots(3, 3, figsize=(12, 9))
        axes = axes.flatten()
        for ax, (frame_index, mask, bbox) in zip(axes, selected):
            image = Image.open(os.path.join(frames_dir, frame_names[frame_index])).convert("RGB")
            if resize_factor != 1.0:
                image = image.resize((int(image.width * resize_factor), int(image.height * resize_factor)))
            img_np = np.array(image)
            bbox_scaled = tuple(int(c * resize_factor) for c in bbox)

            pad = 5
            x_min = max(bbox_scaled[0] - pad, 0)
            y_min = max(bbox_scaled[1] - pad, 0)
            x_max = min(bbox_scaled[2] + pad, img_np.shape[1])
            y_max = min(bbox_scaled[3] + pad, img_np.shape[0])

            ax.imshow(img_np)
            ax.add_patch(Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                   linewidth=1, edgecolor='red', facecolor='none'))
            ax.set_title(f"Frame {frame_index}")
            ax.axis("off")

        fig.suptitle(f"Mask #{mask_num} | Frames {min(frame_indices)}-{max(frame_indices)}", fontsize=14)
        plt.tight_layout()
        plt.show()
        gc.collect()


display_mask_grid(
    n_per_mask=N_PER_MASK,
    resize_factor=RESIZE_FACTOR,
    mask_dir=SEGMENTS_DIR,
    frames_dir=FRAMES_DIR,
    frame_names=frame_names,
    image_shape=image_shape,
)

# Mask pruning

Keep only the tracks listed in `KEEP_KEYS` (configuration cell).

In [ ]:
all_masks = {}
for fname in sorted(os.listdir(SEGMENTS_DIR)):
    if fname.endswith(".npy"):
        all_masks[fname] = np.load(os.path.join(SEGMENTS_DIR, fname), allow_pickle=True).item()

available_ids = sorted({k for data in all_masks.values() for k in data})
keep_set = set(KEEP_KEYS)
if not (keep_set & set(available_ids)):
    raise ValueError(
        f"KEEP_KEYS {sorted(keep_set)} matches none of the tracks found: {available_ids}.\n"
        f"Look at the review grid above and set KEEP_KEYS in the configuration cell."
    )
if keep_set - set(available_ids):
    print(f"Warning: KEEP_KEYS {sorted(keep_set - set(available_ids))} not found (available: {available_ids})")

os.makedirs(PRUNED_DIR, exist_ok=True)
for fname, data in tqdm(all_masks.items(), desc="Pruning"):
    np.save(os.path.join(PRUNED_DIR, fname), {k: v for k, v in data.items() if k in keep_set})
print(f"Kept tracks {sorted(keep_set & set(available_ids))} of {available_ids}")
del all_masks
gc.collect()

# Visualization

Overlay the pruned masks on every frame and encode a video.

In [ ]:
os.makedirs(OVERLAY_DIR, exist_ok=True)

COLORS = [  # BGR
    (0, 0, 255),     # red
    (0, 255, 0),     # green
    (255, 0, 255),   # magenta
    (255, 0, 0),     # blue
    (0, 255, 255),   # yellow
    (255, 165, 0),   # orange
    (128, 0, 128),   # purple
]
mask_colors = {k: COLORS[n % len(COLORS)] for n, k in enumerate(sorted(KEEP_KEYS))}


def save_frame_with_masks_cv2(frame_path, masks, output_path):
    """Overlay each mask in its own colour on the frame and save with OpenCV."""
    image = cv2.imread(frame_path)
    for key, mask in masks.items():
        overlay = np.zeros_like(image, dtype=np.uint8)
        overlay[mask] = mask_colors.get(key, COLORS[0])
        image = cv2.addWeighted(image, 1.0, overlay, 0.4, 0)
    cv2.imwrite(output_path, image)


for frame_idx in tqdm(range(len(frame_names)), desc="Overlaying masks"):
    mask_path = os.path.join(PRUNED_DIR, f'frame_{frame_idx}.npy')
    masks = {}
    if os.path.exists(mask_path):
        polygons = np.load(mask_path, allow_pickle=True).item()
        masks = {k: polygons_to_mask(v, image_shape) for k, v in polygons.items()}

    save_frame_with_masks_cv2(
        os.path.join(FRAMES_DIR, frame_names[frame_idx]),
        masks,
        os.path.join(OVERLAY_DIR, f'{frame_idx:05d}.jpg'),
    )
    if frame_idx % 100 == 0:
        gc.collect()

In [ ]:
output_fps = SOURCE_FPS or 30
run_cmd([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-framerate", f"{output_fps:g}", "-start_number", "0", "-i", f"{OVERLAY_DIR}/%05d.jpg",
    "-vf", "scale=ceil(iw/2)*2:ceil(ih/2)*2", "-c:v", "libx264", "-pix_fmt", "yuv420p", OUTPUT_VIDEO,
])
print(f"Saved {OUTPUT_VIDEO} ({os.path.getsize(OUTPUT_VIDEO) / 1e6:.1f} MB)")

In [ ]:
from base64 import b64encode
from IPython.display import HTML

# Inline preview (skipped for large files - the video is saved to Drive at the final sync step anyway)
if os.path.getsize(OUTPUT_VIDEO) > 25e6:
    print("Video is large; skipping the inline preview.")
    preview = None
else:
    with open(OUTPUT_VIDEO, "rb") as f:
        data_url = "data:video/mp4;base64," + b64encode(f.read()).decode()
    preview = HTML(f"""<video width="1024" height="768" controls><source src="{data_url}" type="video/mp4"></video>""")
preview

# Regression check (optional)

Compares this run's pruned masks with the pruned masks from the original notebook (`BASELINE_PRUNED_DIR`, read-only).
Tracks are compared as the union of all kept masks per frame, so track IDs do not need to match. Skipped if no baseline exists.

In [ ]:
def _union_mask(npy_path):
    data = np.load(npy_path, allow_pickle=True).item()
    union = np.zeros(image_shape[:2], dtype=bool)
    for polygons in data.values():
        union |= polygons_to_mask(polygons, image_shape)
    return union


regression_result = None
if not os.path.isdir(BASELINE_PRUNED_DIR):
    print(f"No baseline found at {BASELINE_PRUNED_DIR} - skipping regression check.")
else:
    base_files = {f for f in os.listdir(BASELINE_PRUNED_DIR) if f.endswith(".npy")}
    new_files = {f for f in os.listdir(PRUNED_DIR) if f.endswith(".npy")}
    dice_scores, both_empty = [], 0
    for fname in sorted(base_files & new_files):
        a = _union_mask(os.path.join(BASELINE_PRUNED_DIR, fname))
        b = _union_mask(os.path.join(PRUNED_DIR, fname))
        if not a.any() and not b.any():
            both_empty += 1
            continue
        dice_scores.append(2.0 * np.logical_and(a, b).sum() / (a.sum() + b.sum()))

    if not dice_scores:
        print("No overlapping non-empty frames to compare.")
    else:
        mean_dice, min_dice = float(np.mean(dice_scores)), float(np.min(dice_scores))
        passed = mean_dice >= REGRESSION_MIN_DICE
        regression_result = {
            "frames_compared": len(dice_scores), "frames_both_empty": both_empty,
            "frames_only_in_baseline": len(base_files - new_files), "frames_only_in_new": len(new_files - base_files),
            "mean_dice": round(mean_dice, 4), "min_dice": round(min_dice, 4),
            "threshold": REGRESSION_MIN_DICE, "passed": passed,
        }
        print(("PASS" if passed else "FAIL"), regression_result)

# Save results to Drive

In [ ]:
sync_run_to_drive("final")
print("\nFiles in", RUN_DRIVE)
for name in sorted(os.listdir(RUN_DRIVE)):
    print(f"  {name}  ({os.path.getsize(os.path.join(RUN_DRIVE, name)) / 1e6:.2f} MB)")
print("\nTimings (s):", timings)